## loading package

In [1]:
#| export


import os
import io
import re
import random
import base64
from io import BytesIO

import time
from datetime import timedelta

import numpy as np

import matplotlib.pyplot as plt

import torch
import torch.nn.functional as F

from IPython.display import SVG, display, Image

from PIL import Image

import cv2
import pandas as pd
import json


from diffusers import StableDiffusionPipeline
from transformers import AutoProcessor, AutoModel



In [2]:
#| export

import vtracer

# import torch._dynamo
# torch._dynamo.config.suppress_errors = True


## Competition Metric Helpers

We also want to evaluate metrics of the original bitmap before converting to svg. Let’s implement it using [metric package](https://www.kaggle.com/code/jiazhuang/svg-image-fidelity).

In [3]:
import metric
import numpy as np
import statistics
import pandas as pd

def image_resize(image, size=(384, 384)):
    return image.convert('RGB').resize(size)

def bitmap_score_instance_impl(multiple_choice_qa, image, random_seed=42):
    rng = np.random.RandomState(random_seed)
    group_seed = rng.randint(0, np.iinfo(np.int32).max)
    image_processor = metric.ImageProcessor(image=image_resize(image), seed=group_seed).apply()
    image = image_processor.image.copy()
    questions = multiple_choice_qa['question']
    choices = multiple_choice_qa['choices']
    answers = multiple_choice_qa['answer']
    aesthetic_score = metric.aesthetic_evaluator.score(image)
    vqa_score = metric.vqa_evaluator.score(questions, choices, answers, image)
    image_processor.reset().apply_random_crop_resize().apply_jpeg_compression(quality=90)
    ocr_score = metric.vqa_evaluator.ocr(image_processor.image)
    instance_score = metric.harmonic_mean(vqa_score, aesthetic_score, beta=0.5) * ocr_score
    return instance_score, vqa_score, ocr_score, aesthetic_score

def bitmap_score_instance(multiple_choice_qa, image, random_seed=42):
    is_single = not isinstance(image, list)
    if is_single:
        multiple_choice_qa = [multiple_choice_qa]
        image = [image]
    
    assert len(multiple_choice_qa) == len(image)

    results = []
    score_df = []
    for one_image, one_multiple_choice_qa in zip(image, multiple_choice_qa, strict=True):
        instance_score, vqa_score, ocr_score, aesthetic_score = bitmap_score_instance_impl(one_multiple_choice_qa, one_image, random_seed=42)
        results.append(instance_score)
        score_df.append([instance_score, vqa_score, ocr_score, aesthetic_score])

    fidelity = statistics.mean(results)
    score_df = pd.DataFrame(score_df, columns=['competition_score', 'vqa_score', 'ocr_score', 'aesthetic_score'])
    if is_single:
        return score_df.iloc[0].to_dict()
    else:
        return float(fidelity), score_df

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Q1: Is there a red circle?
Predicted Answer: yes
Correct Answer  : yes
Probabilities   : {'yes': 0.901214732510082, 'no': 0.09878526748991795}

Q2: What shape is present?
Predicted Answer: circle
Correct Answer  : circle
Probabilities   : {'square': 0.00039966567346408317, 'circle': 0.9992734001747942, 'triangle': 0.0001767018281842993, 'hexagon': 0.0001502323235575014}

decoded text =  100
Q1: Is there a red circle?
Predicted Answer: yes
Correct Answer  : yes
Probabilities   : {'yes': 0.901214732510082, 'no': 0.09878526748991795}

Q2: What shape is present?
Predicted Answer: circle
Correct Answer  : circle
Probabilities   : {'square': 0.00039966567346408317, 'circle': 0.9992734001747942, 'triangle': 0.0001767018281842993, 'hexagon': 0.0001502323235575014}

decoded text =  100
Does <image> portray "SVG illustration of a red circle" without any lettering? Answer yes or no.
Q1: Is there a red circle?
Predicted Answer: yes
Correct Answer  : yes
Probabilities   : {'yes': 0.901214732510082,

## Load Stable Diffusion

In [4]:
from diffusers import FluxTransformer2DModel
import torch
from diffusers import BitsAndBytesConfig as DiffusersBitsAndBytesConfig, FluxTransformer2DModel, FluxPipeline
from transformers import BitsAndBytesConfig as BitsAndBytesConfig, T5EncoderModel
from diffusers import AutoencoderKL, AutoencoderTiny
from diffusers.hooks import apply_group_offloading
from optimum.quanto import freeze, qfloat8, quantize, qint8, qint4


# transformer = 'flux/flux1-schnell-Q4_K_S.gguf'
# vae = 'flux/ae.safetensors'
# t5_path = 't5_fp8'
flux_path =  "black-forest-lab/FLUX.1-schnell" 
vae_path  = 'taef1'

# ==========================load transformer=========================================
print('loading transformer...')
nf4_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)
transformer = FluxTransformer2DModel.from_pretrained(
    flux_path,
    subfolder="transformer",
    quantization_config=nf4_config,
    torch_dtype=torch.bfloat16
)

# transformer = FluxTransformer2DModel.from_single_file(
#     flux_fp8, 
#     torch_dtype=torch.bfloat16,
#     config = flux_path + '/transformer',
# )
# quantize(transformer, weights=qint4)
# freeze(transformer)

# ==========================load t5=========================================
print('loading t5...')
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16
)
t5_nf4 = T5EncoderModel.from_pretrained("t5-nf4", torch_dtype=torch.bfloat16, quantization_config=bnb_config)


# ==========================load vae=========================================
print('loading vae...')
vae = AutoencoderTiny.from_pretrained(vae_path, torch_dtype=torch.bfloat16)

# ==========================load pipeline=========================================
print('loading pipeline...')
base = FluxPipeline.from_pretrained(
    pretrained_model_name_or_path=flux_path,
    transformer=transformer,
    vae = vae,
    text_encoder_2 = t5_nf4,
    torch_dtype=torch.bfloat16,
    local_files_only=True,
)
base = base.to("cuda:0")
# base.vae.enable_tiling()
base.enable_vae_slicing()
# base.enable_model_cpu_offload() # this is slower than group-offloading, and the problem is it automatically onload to cuda:0

torch.cuda.empty_cache()

# apply_group_offloading(
#     base.text_encoder_2, 
#     offload_device=torch.device("cpu"),
#     onload_device=torch.device("cuda:1"),
#     offload_type="leaf_level",
#     use_stream=True,
#     record_stream=True,
# )


loading transformer...


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

loading t5...


c:\Users\YongsanHuang\.conda\envs\sd\Lib\site-packages\transformers\quantizers\auto.py:212: UserWarning: You passed `quantization_config` or equivalent parameters to `from_pretrained` but the model you're loading already has a `quantization_config` attribute. The `quantization_config` from the model will be used.
  warnings.warn(warning_msg)


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

The config attributes {'block_out_channels': [64, 64, 64, 64]} were passed to AutoencoderTiny, but are not expected and will be ignored. Please verify your config.json configuration file.


loading vae...
loading pipeline...


Loading pipeline components...:   0%|          | 0/7 [00:00<?, ?it/s]

You set `add_prefix_space`. The tokenizer needs to be converted from the slow tokenizers
Expected types for vae: (<class 'diffusers.models.autoencoders.autoencoder_kl.AutoencoderKL'>,), got <class 'diffusers.models.autoencoders.autoencoder_tiny.AutoencoderTiny'>.


## load llm

In [5]:
#| export
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()
from transformers import AutoTokenizer, BitsAndBytesConfig, AutoModelForCausalLM, GenerationConfig
import torch

model_id = r"Qwen3-1.7B-unsloth-bnb-4bit"

quantization_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.bfloat16)

llm_model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=quantization_config
).eval().to("cuda:0")

llm_tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)

gen_config = GenerationConfig(
    do_sample=True,
    temperature=0.8,
    top_p=0.95,
    max_new_tokens=128,
    pad_token_id=llm_tokenizer.eos_token_id,
    eos_token_id=llm_tokenizer.eos_token_id,
)


## classify prompt

In [6]:
#| export
def classify_prompt(raw_prompt: str) -> str:
    messages = [
        {
            "role": "system",
            "content": """You are a classification assistant. Given a short prompt and based on the main subjects it describes, classify it into one of the following most related categories:

1 = landscape (scenery, nature, outdoor environments)
2 = fashion (clothing, accessories, wearable items, anything human can wear)
3 = geometry (abstract shapes, patterns, colors, layouts)

Your return should be one number(1, 2, or 3) only. Return nothing else.
"""
        },
        {
            "role": "user",
            "content": """Example: 
prompt:'a purple forest at dusk' -> category: 1
prompt: 'gray wool coat with a faux fur collar' -> category: 2
prompt: 'khaki triangles and azure crescents' -> category: 3

Prompt: 'a lighthouse overlooking the ocean'. category:
"""
        },
        {
            "role": "assistant",
            "content": "1"
        },
        {
            "role": "user",
            "content": f"prompt: {raw_prompt}. category:"
        },
    ]

    try:
        inputs = llm_tokenizer.apply_chat_template(
            messages,
            enable_thinking=False,
            add_generation_prompt=True,
            tokenize=True,
            return_dict=True,
            return_tensors="pt",
        ).to(llm_model.device).to(torch.bfloat16)
        # print(inputs)
        # print(type(inputs))
    
        with torch.inference_mode():
            outputs = llm_model.generate(**inputs, generation_config=gen_config)
    
        outputs = outputs[:, inputs.input_ids.shape[1]:]
        outputs = llm_tokenizer.batch_decode(outputs, skip_special_tokens=True)
    
        # print(outputs)
        # print(type(outputs))
    
        # outputs[0] is the full chat + elaboration
        return outputs[0].strip()
    
    except Exception as e:
        print(f"[Prompt fallback] Failed to elaborate prompt: {e}")
        output = '0'
        return output

## elaborate prompt_landscape

In [7]:
#| export
def elaborate_prompt_scenery(raw_prompt: str) -> str:
    messages = [
        {
            "role": "system",
            "content": """You are a skilled prompt engineer for image generation. You need to convert a short raw prompt into a stylized descriptive sentence suitable for generating sticker in Flux.1.
The prompt format is: A stylized ...., rendered in.... Simplified geometric....silhouettes....
Some rules you must follow when crafting the prompt:
1. Clearly define the subject and its attributes, and relationship between different elements. No new elements should be added.
2. Only return the elaborated prompt with nothing else.
"""
        },
        {
            "role": "user",
            "content": """Elaborate a prompt for me so that I can use it for generating vector-art image. 
Some examples:
'a purple forest at dusk' -> 'A stylized abstract purple forest at dusk, rendered in soft pastel hues of lavender, indigo, and pale gold. Simplified geometric tree silhouettes with flowing, organic shapes, glowing gradient light effects in the sky.'
'gray wool coat with a faux fur collar' -> 'A stylized abstract gray wool coat with a faux fur collar on a lady, rendered in soft muted tones of charcoal, dove gray, and ivory. Simplified geometric tailoring defines the coat’s clean silhouette, while the collar is illustrated with flowing, feathery textures in layered light beige and cream, evoking warmth and refined elegance.'
'khaki triangles and azure crescents' -> 'A stylized abstract arrangement of khaki triangles and azure crescents, rendered in soft pastel hues of sand, olive, and sky blue. Simplified geometric forms are carefully balanced in a rhythmic composition, with subtle gradients and overlapping shapes creating a sense of motion and visual harmony against a light, neutral backdrop.'

Raw prompt: 'a lighthouse overlooking the ocean'. Your elaborated prompt:
"""
        },
        {
            "role": "assistant",
            "content": "A stylized lighthouse perched on a rocky cliff above a calm ocean, rendered in soft tones of misty blue, slate gray, and warm ivory. Simplified geometric forms and gentle gradients evoke a tranquil seascape at dawn, with minimal waves and a faint beam of light sweeping across the horizon."
        },
        {
            "role": "user",
            "content": f"raw prompt: {raw_prompt}. Your elaborated prompt:"
        },
    ]

    try:
        inputs = llm_tokenizer.apply_chat_template(
            messages,
            enable_thinking=False,
            add_generation_prompt=True,
            tokenize=True,
            return_dict=True,
            return_tensors="pt",
        ).to(llm_model.device).to(torch.bfloat16)
        # print(inputs)
        # print(type(inputs))
    
        with torch.inference_mode():
            outputs = llm_model.generate(**inputs, generation_config=gen_config)
    
        outputs = outputs[:, inputs.input_ids.shape[1]:]
        outputs = llm_tokenizer.batch_decode(outputs, skip_special_tokens=True)
        output = outputs[0].strip()
        output = output + 'The composition uses smooth color blocks, subtle line art, and a dreamlike, flattened perspective to emphasize elegance and simplicity. Inspired by minimalist vector art, with vibrant color and cinematic lighting. The overall atmosphere is tranquil yet powerful, with strong color contrasts, 8K, masterpiece, trending at artstation.'
    
        # print(outputs)
        # print(type(outputs))
    
        # outputs[0] is the full chat + elaboration
        return output
    
    except Exception as e:
        print(f"[Prompt fallback] Failed to elaborate prompt: {e}")
        output = f'Create a stylized geometric illustration of {raw_prompt} with geometric form and abstract silhouettes. The composition uses smooth color blocks, subtle line art, and a dreamlike, flattened perspective to emphasize elegance and simplicity. Inspired by minimalist vector art, with vibrant color and cinematic lighting. The overall atmosphere is tranquil yet powerful, with strong color contrasts, 8K, masterpiece, trending at artstation.'
        return output

## elaborate prompt_geometry

In [8]:
#| export
def elaborate_prompt_geo(raw_prompt: str) -> str:
    messages = [
        {
            "role": "system",
            "content": """You are a skilled prompt engineer for image generation. You need to convert a short raw prompt into a stylized descriptive sentence suitable for generating sticker in Flux.1."
The prompt format is: A sticker depicting...., shown in a flat 2D front view against a pure white background.... with no shadows, reflection or depth, evoking a vectorized, minimalist design style."
Some rules you must follow when crafting the prompt:
1. Clearly and precisely define the shape, color and relationship between each shape(defalut is no overlaping between different shapes)
2. Define the amount of each shapes. If it is single, call it one; if it is plural, give it a reasonable amount(default amount is three)
3. Pure white background, but if there is any shape in white color already, pick another neutral dark color as background.
4. Shape should be bold and clear and visible. For example, a thread should be a thick thread to be visible and not covered by other shape.
Only return the elaborated prompt with nothing else.
"""
        },
        {
            "role": "user",
            "content": """Elaborate a prompt for me so that I can use it for generating sticker. 
Some examples:
'khaki triangles and azure crescents' -> 'A sticker depicting three khaki triangles and two azure crescents, shown in a flat 2D front view against a pure white background. The triangles have bold, solid khaki fills with sharp corners, arranged in a loose cluster with slight angular variation, while the crescents are smooth, thick arcs in bright azure, placed nearby in curved contrast, with no shadows, reflection or depth, evoking a vectorized, minimalist design style.'
'purple pyramids spiraling around a bronze cone' -> 'A sticker depicting three purple pyramids spiraling around a bronze cone, shown in a flat 2D front view against a pure white background. The cone stands central with a solid flat bronze fill and clean edges, while three bold purple pyramids orbit in a loose spiral pattern, evenly spaced and clearly defined, with no shadows, reflection or depth, evoking a vectorized, minimalist design style.'
'crimson rectangles forming a chaotic grid' -> 'A sticker depicting seven crimson rectangles forming a chaotic grid, shown in a flat 2D front view against a pure white background. The rectangles have bold, solid crimson fills with clean black outlines, scattered and overlapping at irregular angles, creating a fragmented, unbalanced pattern, with no shadows, reflection or depth, evoking a vectorized, minimalist design style.'

Raw prompt: 'magenta trapezoids layered on a transluscent silver sheet'. Your elaborated prompt:
"""
        },
        {
            "role": "assistant",
            "content": "A sticker depicting three magenta trapezoids layered on one translucent grey color sheet, shown in a flat 2D front view against a pure white background. The trapezoids have bold, solid fills with sharp edges, stacked diagonally with slight overlap, while the grey color sheet appears as a soft, semi-transparent base, with no shadows, reflection or depth, evoking a vectorized, minimalist design style."
        },
        {
            "role": "user",
            "content": f"raw prompt: {raw_prompt}. Your elaborated prompt:"
        },
    ]

    try:
        inputs = llm_tokenizer.apply_chat_template(
            messages,
            enable_thinking=False,
            add_generation_prompt=True,
            tokenize=True,
            return_dict=True,
            return_tensors="pt",
        ).to(llm_model.device).to(torch.bfloat16)
        # print(inputs)
        # print(type(inputs))
    
        with torch.inference_mode():
            outputs = llm_model.generate(**inputs, generation_config=gen_config)
    
        outputs = outputs[:, inputs.input_ids.shape[1]:]
        outputs = llm_tokenizer.batch_decode(outputs, skip_special_tokens=True)
    
        # print(outputs)
        # print(type(outputs))
    
        # outputs[0] is the full chat + elaboration
        return outputs[0].strip()
    
    except Exception as e:
        print(f"[Prompt fallback] Failed to elaborate prompt: {e}")
        output = f'Create a sticker of {raw_prompt} shown in a flat 2D front view against a pure white background.There is no shadows, reflection or depth, evoking a vectorized, minimalist design style.'
        return output

In [9]:
%%time
prompt = 'a purple silk scarf with tassel trim'
prompt = elaborate_prompt_geo(prompt)
print(prompt)

Attempting to cast a BatchEncoding to type torch.bfloat16. This is not supported.
`generation_config` default values have been modified to match model-specific defaults: {'max_length': 40960, 'top_k': 20, 'bos_token_id': 151643}. If this is not desired, please set these values explicitly.


A sticker depicting two purple silk scarves with tassel trim, shown in a flat 2D front view against a pure white background. The scarves have bold, solid purple fills with clean edges, arranged side by side with slight overlap, while the tassel trim is thick, bold, and in contrasting color, placed at the ends of the scarves, with no shadows, reflection or depth, evoking a vectorized, minimalist design style.
CPU times: total: 25.3 s
Wall time: 44.3 s


In [10]:
#| export
def generate_bitmaps(prompt, prompt_2 = '', negative_prompt=""):
    
    # run both experts
    images = base(
        prompt=prompt,
        prompt_2 = prompt_2 if prompt_2 != None else prompt,
        negative_prompt = negative_prompt,
        width=512,
        height=512,
        num_inference_steps=1,
        guidance_scale = 0,
        num_images_per_prompt=3,
    ).images
    
    return images



## Load Data

In [12]:
import pandas as pd
import json
train_df = pd.read_csv('train.csv')
train_question_df = pd.read_parquet('questions.parquet')

train_question_df = train_question_df.groupby('id').apply(lambda df: df.to_dict(orient='list'))
train_question_df = train_question_df.reset_index(name='qa')

train_question_df['question'] = train_question_df.qa.apply(lambda qa: json.dumps(qa['question'], ensure_ascii=False))

train_question_df['choices'] = train_question_df.qa.apply(
    lambda qa: json.dumps(
        [x.tolist() for x in qa['choices']], ensure_ascii=False
    )
)

train_question_df['answer'] = train_question_df.qa.apply(lambda qa: json.dumps(qa['answer'], ensure_ascii=False))

train_df = pd.merge(train_df, train_question_df, how='left', on='id')

train_df['multiple_choice_qa'] = train_df.apply(
    lambda r: {
    'question': json.loads(r.question),
    'choices': json.loads(r.choices),
    'answer': json.loads(r.answer)
    },
    axis=1,
)

# train_df.head()

C:\Users\YongsanHuang\AppData\Local\Temp\ipykernel_15132\3134480291.py:6: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  train_question_df = train_question_df.groupby('id').apply(lambda df: df.to_dict(orient='list'))


# Image -> SVG

* Did a bunch of work here trying to get good results..

In [ ]:
import re
import vtracer
from IPython.display import SVG, display, Image
from PIL import Image as PILImage
input_path = "output.png"
output_path = "output.svg"

## new img2svg

In [ ]:
#| export
import re
def convert_paths_to_polygons(svg_code: str, size: int) -> str:
    # Find all path elements
    scale_factor = 384 / size

    path_pattern = re.compile(
        r'<path[^>]*d="([^"]+)"[^>]*fill="([^"]+)"[^>]*transform="translate\(([^)]+)\)"[^>]*/?>'
    )
    paths = path_pattern.findall(svg_code)

    polygons = []

    for d_content, fill_color, translate in paths:
        tx, ty = map(float, translate.split(','))
        points = []
        tokens = re.findall(r'[MLZmlz]|-?\d+\.?\d*,\-?\d+\.?\d*', d_content.strip())
        for token in tokens:
            if token in {'M', 'L', 'Z', 'm', 'l', 'z'}:
                continue
            x_str, y_str = token.split(',')
            x = int(round(float(x_str) + tx))
            y = int(round(float(y_str) + ty))
            points.append(f"{x},{y}")

        if points:
            polygon = f'<polygon points="{" ".join(points)}" fill="{fill_color}"/>'
            polygons.append(polygon)

    # Build the compact SVG
    new_svg = (
        f'<svg width="384" height="384" viewBox="0 0 384 384"><g transform="scale({scale_factor})">'
        + "".join(polygons)
        + '</g></svg>'
    )

    return new_svg


def fix_svg_size(svg_code: str, target_size: int = 384) -> str:
    """
    Update the <svg> tag's width and height to match target size.
    """
    # Replace width attribute
    svg_code = re.sub(
        r'(width\s*=\s*")([^"]+)(")',
        lambda m: f'{m.group(1)}{target_size}{m.group(3)}',
        svg_code,
        count=1
    )
    # Replace height attribute
    svg_code = re.sub(
        r'(height\s*=\s*")([^"]+)(")',
        lambda m: f'{m.group(1)}{target_size}{m.group(3)}',
        svg_code,
        count=1
    )
    return svg_code


def bitmap_to_svg_layered(img, input_path, output_path):
    default_svg = """<svg width="384" height="384" viewBox="0 0 384 384"><circle cx="50" cy="50" r="40" fill="red" /></svg>"""
    # Step 1: Resize the input image to 256x256
    # 256x256 is helpful bc each length od path is shorter, so with the same max_svg_length, we can have more paths(more color)
    size = 128
    # img = img.resize((size,size), PILImage.LANCZOS)
    img.save(input_path)

    # Step 2: Convert the resized image to SVG
    max_svg_length = 9996  # target length limit

    # Define injection to prevent OCR hallucination
    injection_a1 = '\n<path d="M5 374 L10 364 L15 374 M7 368 L13 368" stroke="white"/>'      # Bottom-left A
    injection_a2 = '\n<path d="M364 30 L370 20 L376 30 M367 25 L373 25" stroke="white"/>'         # Top-right A
    injection = injection_a1 + injection_a2
    injection_length = len(injection)

    # Binary search for best layer_difference
    low = 1
    high = 200
    best_svg_code = None
    best_layer_difference = 1

    while low <= high:
        layer_difference = (low + high) // 2

        vtracer.convert_image_to_svg_py(
            input_path,
            output_path,
            colormode='color',        
            hierarchical='stacked',  
            mode='polygon',          
            filter_speckle=1,         
            color_precision=8,      
            layer_difference=layer_difference, 
            corner_threshold=10,     
            length_threshold=10,     
            max_iterations=10,        
            splice_threshold=10,     
            path_precision=3         
        )

        # Step 3: Read and display the SVG
        try:
            with open(output_path, "r", encoding="utf-8") as f:
                svg_code = f.read()
        except UnicodeDecodeError:
            # Bad SVG output (probably corrupted), try again
            high = layer_difference - 1
            continue  # go back to binary search

        # Clean the first two lines if present
        lines = svg_code.splitlines()
        removed_length = 0
        if lines and lines[0].strip().startswith('<?xml'):
            removed_length += len(lines[0]) + 1  # +1 for newline
            lines = lines[1:]
        if lines and lines[0].strip().startswith('<!--'):
            removed_length += len(lines[0]) + 1  # +1 for newline
            lines = lines[1:]
        svg_code = "\n".join(lines)

        svg_code = svg_code.replace(
            '<svg ',
            '<svg viewBox="0 0 384 384" ', #this is 20 bytes more
            1  # only replace first occurrence
        )
        
        # remove version and xmlns attributes
        svg_code = re.sub(r'\s*version="[^"]*"', '', svg_code)
        svg_code = re.sub(r'\s*xmlns="[^"]*"', '', svg_code)

        # use polygon instead of path
        svg_code = convert_paths_to_polygons(svg_code, size)
            
        # Correct length check: give credit for removed lines
        if len(svg_code) + injection_length <= max_svg_length:
            best_svg_code = svg_code
            best_layer_difference = layer_difference
            high = layer_difference - 1  # search for even more detail
        else:
            low = layer_difference + 1  # simplify more

    try:
        svg_code = fix_svg_size(best_svg_code, target_size=384) # doesnt change length
        # Inject fake letter path to prevent OCR hallucination
        svg_code = svg_code.replace("</svg>", injection + "</svg>")
    except Exception as e:
        print(f"Error fixing SVG size: {e}")
        svg_code = default_svg

    print(f'{best_layer_difference=}')
    return svg_code


## SD evaluator

In [ ]:
#| export
from PIL import Image
import ast

def generate_qa_llm(prompt: str) -> str:
    messages = [
        {
            "role": "system",
            "content": """You reword text into 3 questions.
Each question must be supported exactly in provided text. The best answer is always from the text. Ask meaningful questions.
Return a dict of this format: {"question": [list of 3 questions], "choices":[[multiple choices for question 1],[multiple choices for question2], [multiple choices for question2]], "answer":[list of one correct answer to each question]}
Only return a dict without nothing else. The dict must be a valid dict with correct format.
"""
        },
        {
            "role": "user",
            "content": """Create a series of questions, choices and answers based on my input prompt.
Some examples:
'a purple forest at dusk' -> {"question": ["What is the main setting of the image?", "Is there anything purple in the image?", "What time of day is suggested in the image?"], "choices": [["beach", "desert", "forest", "mountain"], ["no", "yes"], ["dawn", "dusk", "midday", "midnight"]], "answer": ["forest", "yes", "dusk"]}
'gray wool coat with a faux fur collar' -> {"question": ["What color is the coat?", "What part of the coat has faux fur?", "What material is the coat made of?"], "choices": [["blue", "brown", "gray", "red"], ["collar", "hem", "pockets", "sleeves"], ["cotton", "leather", "silk", "wool"]], "answer": ["gray", "collar", "wool"]}
'crimson rectangles forming a chaotic grid' -> {"question": ["Is the grid's arrangement chaotic?", "What is the color of the rectangles", "Are the primary shapes rectangles?"], "choices": [["no", "yes"], ["crimson", "green", "orange", "pueple"], ["no", "yes"]], "answer": ["yes", "no", "yes"]}

Prompt: magenta trapezoids layered on a transluscent silver sheet. Your QA dict:
"""
        },
        {
            "role": "assistant",
            "content": """{"question": ["Which word describes the silver sheet's ability to let light through?", "Are the trapezoids layered on something?", "What shape are the magenta objects?"], "choices": [["opaque", "reflective", "solid", "translucent"], ["no", "yes"], ["circles", "stars", "trapezoids", "triangles"]], "answer": ["translucent", "yes", "trapezoids"]}"""
        },
        {
            "role": "user",
            "content": f"Prompt: {prompt}. Your QA dict(Only return a dict without nothing else):"
        },
    ]

    try:
        inputs = llm_tokenizer.apply_chat_template(
            messages,
            enable_thinking=False,
            add_generation_prompt=True,
            tokenize=True,
            return_dict=True,
            return_tensors="pt",
        ).to(llm_model.device).to(torch.bfloat16)
        # print(inputs)
        # print(type(inputs))
    
        with torch.inference_mode():
            outputs = llm_model.generate(**inputs, generation_config=gen_config)
    
        outputs = outputs[:, inputs.input_ids.shape[1]:]
        outputs = llm_tokenizer.batch_decode(outputs, skip_special_tokens=True)
        outputs = outputs[0].strip()
        qa = ast.literal_eval(outputs)
        
        return qa

    except Exception as e:
        print(f"[Prompt fallback] Failed to create qa: {e}")
        return generate_qa(prompt)

def generate_qa(prompt: str) -> dict:
    """
    Generate VQA-style question-answer dict based on a given prompt.
    Includes yes/no questions and one clarity rating.
    """
    qa = {
        'question': [
            f"Is the main topic of this image {prompt}?",
            f"Are all elements of {prompt} clearly displayed?",
            f"How identifiable is {prompt} in this image?"
        ],
        'choices': [
            ['no', 'yes'],
            ['no', 'yes'],
            ['not at all', 'acceptably', 'very clear']
        ],
        'answer': [
            'yes',   
            'yes',   
            'very clear' 
        ]
    }
    return qa
    
def image_resize(image, size=(384, 384)):
    return image.convert('RGB').resize(size)

def get_score(sample, qa, ocr=False):
    # If sample is a string, treat as SVG and convert to image
    rng = np.random.RandomState(42)
    group_seed = rng.randint(0, np.iinfo(np.int32).max)
    if isinstance(sample, str):
        image = metric.svg_to_png(sample)
    else:
        image = sample
    
    image_processor = metric.ImageProcessor(image=image_resize(image), seed=group_seed).apply()
    image = image_processor.image.copy()
    aesthetic_score = metric.aesthetic_evaluator.score(image)
    questions = qa['question']
    choices = qa['choices']
    answers = qa['answer']
    vqa_score = metric.vqa_evaluator.score(questions, choices, answers, image)

    if ocr:
        image_processor.reset().apply_random_crop_resize().apply_jpeg_compression(quality=90)
        ocr_score = metric.vqa_evaluator.ocr(image_processor.image)
    else:
        ocr_score = 1.0

    instance_score = metric.harmonic_mean(vqa_score, aesthetic_score, beta=0.5) * ocr_score

    return instance_score, aesthetic_score, ocr_score, vqa_score



## Implement the package Model class

In [ ]:
#| export
import time
class Model:
    # def __init__(self):
    #     self.default_svg = """<svg width="384" height="384" viewBox="0 0 384 384"><circle cx="50" cy="50" r="40" fill="red" /></svg>"""
    #     self.prompt_prefix = "a stylized digital painting presenting a"
    #     self.prompt_suffix = " from distance. The painting promote vector-art aesthetic, in watercolor art style with vibrant and clean background. The overall atmosphere is tranquil yet powerful, raw-photo hyper-detail, 4K, cinematic lighting, award-winning, masterpiece."
    #     self.negative_prompt = 'text, logo, mirror reflection, high-reflective, lines, deformed, ugly, wrong proportion, low res, bad anatomy, worst quality, low quality, framing, hatching, patterns, outlines'
    #     self.num_attempt = 1

    # def __init__(self):
    #     self.default_svg = """<svg width="384" height="384" viewBox="0 0 384 384"><circle cx="50" cy="50" r="40" fill="red" /></svg>"""
    #     self.prompt_prefix = "A masterpiece (watercolor:1.2) painting of"
    #     self.prompt_suffix = ", soft brushstrokes capturing the tranquil yet powerful atmosphere, vibrant colors blending together."
    #     self.negative_prompt = 'text, logo, deformed, ugly, wrong proportion, low res, worst quality, low quality'
    #     self.num_attempt = 1

    def __init__(self):
        self.default_svg = """<svg width="384" height="384" viewBox="0 0 384 384"><circle cx="50" cy="50" r="40" fill="red" /></svg>"""
        # self.prompt_prefix = ""
        # self.prompt_suffix = "The composition uses smooth color blocks, subtle line art, and a dreamlike, flattened perspective to emphasize elegance and simplicity. Inspired by minimalist vector art, with vibrant color and cinematic lighting. The overall atmosphere is tranquil yet powerful, with strong color contrasts, 8K, masterpiece, trending at artstation."
        self.negative_prompt = 'text, logo, realistic, hyper detail, reflection, deformed, ugly, wrong proportion, low res, worst quality, low quality'
        self.num_attempt = 1

    def gen_bitmaps(self, description):
        prompt = description
        prompt_type = classify_prompt(description)
        print(f'{prompt_type=}')
        if prompt_type == '1':
            description = elaborate_prompt_scenery(description)
        elif prompt_type == '2':
            description = elaborate_prompt_scenery(description)
        elif prompt_type == '3':
            description = elaborate_prompt_geo(description)
        else:
            description = elaborate_prompt_scenery(description)
        prompt_2 = description
        print(f'======= {prompt_2} =======')        
        bitmaps = generate_bitmaps(prompt = prompt, prompt_2 = prompt_2, negative_prompt = self.negative_prompt)
        return bitmaps

    def predict_impl(self, prompt: str, prompt_2 = '') -> str:
        try:
            qa = generate_qa_llm(prompt)
        except Exception as e:
            qa = generate_qa(prompt)
        print(qa)

        best_score = 0.0
        best_svg = None
        best_img = None
        start_time = time.time()
        for i in range(self.num_attempt):
            if time.time() - start_time > 40:
                print(f"Timeout reached at attempt {i}. Returning current best result.")
                break
            bitmap_list = self.gen_bitmaps(prompt)
            for bitmap in bitmap_list:
                bitmap = bitmap.resize((128, 128), PILImage.LANCZOS)
                display(bitmap)
                instance_score, aesthetic_score, ocr_score, vqa_score = get_score(sample=svg, qa = qa, ocr=False)
                score = instance_score
                print(f'{aesthetic_score =}, {vqa_score=}, {score=}')
                if score >= best_score:
                    best_score = score
                    best_img = bitmap
        print('final score:', best_score)
        print('=======================================================================')

        best_svg = bitmap_to_svg_layered(best_img, input_path, output_path)
        print('svg length:', len(best_svg))
        
        if best_svg is None:
            best_svg = self.default_svg

        return best_svg, best_img

    def predict(self, prompt: str) -> str:
        svg, img = self.predict_impl(prompt)
        return svg


In [ ]:
model = Model()

In [ ]:
%%time
r = train_df.iloc[8]
description = r.description
svg, img = model.predict_impl(description)
display(img)
display(SVG(svg))
print(svg)

In [ ]:
print('='*40)
print(metric.score_instance(r.multiple_choice_qa, svg, random_seed=42))
print('='*40)
print(bitmap_score_instance(r.multiple_choice_qa, img, random_seed=42))

## Evaluate on train dataset (LB prediction!)

In [ ]:
import matplotlib.pyplot as plt
%matplotlib inline

import pandas as pd
from tqdm.auto import tqdm
tqdm.pandas()

train_df['raw_res'] = train_df.description.progress_apply(model.predict_impl)

train_df['svg'] = train_df.raw_res.apply(lambda x: x[0])
train_df['bitmap'] = train_df.raw_res.apply(lambda x: x[1])

train_df['bitmap_score'] = train_df.progress_apply(
    lambda r: bitmap_score_instance(r.multiple_choice_qa, r.bitmap, random_seed=42),
    axis=1,
)

train_df['svg_score'] = train_df.progress_apply(
    lambda r: metric.score_instance(r.multiple_choice_qa, r.svg, random_seed=42),
    axis=1,
)

for r in train_df.itertuples():
    
    b_vqa = r.bitmap_score['vqa_score']
    b_aesthetic = r.bitmap_score['aesthetic_score']
    b_ocr = r.bitmap_score['ocr_score']
    b_score = r.bitmap_score['competition_score']

    
    s_vqa = r.svg_score['vqa_score']
    s_aesthetic = r.svg_score['aesthetic_score']
    s_ocr = r.svg_score['ocr_score']
    s_score = r.svg_score['competition_score']
    
    plt.figure(figsize=(12, 6))
    plt.suptitle(r.description, y=0.93)
    
    plt.subplot(1, 2, 1)
    plt.imshow(np.array(r.bitmap))
    plt.axis('off')
    plt.title(f'bitmap: score={b_score:.2f}, vqa={b_vqa:.2f}, ocr={b_ocr:.2f}, aes={b_aesthetic:.2f}')

    plt.subplot(1, 2, 2)
    plt.imshow(metric.svg_to_png(r.svg))
    plt.axis('off')
    plt.title(f'svg: score={s_score:.2f}, vqa={s_vqa:.2f}, ocr={s_ocr:.2f}, aes={s_aesthetic:.2f}')

## score of train set

In [ ]:
mean_bitmap_score = pd.DataFrame(train_df['bitmap_score'].tolist()).mean(axis=0)
print(mean_bitmap_score)
print('='*20)
mean_svg_score = pd.DataFrame(train_df['svg_score'].tolist()).mean(axis=0)
print(mean_svg_score)

print()
print(f'Original bitmap score: {mean_bitmap_score.competition_score}')
print(f'Final svg score: {mean_svg_score.competition_score}')